# Ozon E-CUP 2026 — CE e5-base stage-A (Colab A100)

Главный калибр: `intfloat/multilingual-e5-base` (278M, MIT), 1 эпоха на 11.2M LLM-пар, без стадии B.

**Ориентиры**: tiny stage-A llm-holdout 0.746 -> LB 0.450; stack 0.762 -> LB 0.461. Цель base: llm-holdout 0.78+.

**Перед запуском (Colab):**
1. Runtime -> Change runtime type -> **A100 GPU** (нужен Colab Pro; если дают только L4 — тоже ок, просто медленнее ~в 2.5 раза).
2. Выполни первую код-ячейку: монтирует Google Drive (туда пишутся чекпойнты — обрыв сессии не теряет прогресс) и качает данные.
3. Runtime -> Run all. Эпоха на A100 ~3-4 часа, на L4 ~8-10.

Чекпойнты: CKPT — лучший по llm-val-fast, CKPT+'.last' — последний. Резюме после обрыва: перезапустить всё, обучение начнётся заново, но лучший чекпойнт на Drive не потеряется (для настоящего resume спроси Егора/Клода).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/ecup', exist_ok=True)
os.makedirs('/content/data', exist_ok=True)
# Данные: положи 4 файла (items.parquet, items_human.parquet, matches.parquet,
# matches_llm.parquet) в MyDrive/ecup_data/ один раз — отсюда они копируются в
# локальный диск колаба (чтение с Drive медленное, копия обязательна).
for f in ['items.parquet', 'items_human.parquet', 'matches.parquet', 'matches_llm.parquet']:
    src = f'/content/drive/MyDrive/ecup_data/{f}'
    dst = f'/content/data/{f}'
    if not os.path.exists(dst):
        assert os.path.exists(src), f'нет {src} — загрузи данные на Drive'
        print('копирую', f)
        !cp "{src}" "{dst}"
print('данные готовы')


In [ ]:
import os, json, re, gc, time, math
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

BASE = "/content/data"  # <-- скачай туда 4 parquet-файла (см. шапку)
MATCHES_PATH = f"{BASE}/matches.parquet"
MATCHES_LLM_PATH = f"{BASE}/matches_llm.parquet"
ITEMS_PATH = f"{BASE}/items.parquet"          # полный, 4.1GB
ITEMS_HUMAN_PATH = f"{BASE}/items_human.parquet"

MODEL_NAME = "intfloat/multilingual-e5-base"   # 278M, MIT
MAX_LEN = 160
BATCH = 192
LR_A, LR_B = 5e-5, 0.0      # base: LR ниже; стадия B отключена
EPOCHS_A, EPOCHS_B = 1, 0
WARMUP = 2000
EVAL_EVERY = 10000          # шагов
CKPT = "/content/drive/MyDrive/ecup/ce_base_ckpt.pt"
OUT_DIR = "/content/drive/MyDrive/ecup/ce_base_final"
SEED = 42

torch.manual_seed(SEED); np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device, torch.cuda.get_device_name(0) if device == "cuda" else "")

## 1. Текст товара
Имя + компактные атрибуты (коды, бренд, вариантные). Эту же функцию потом копируем в сабмит — менять только синхронно!

In [ ]:
KEY_ORDER = ["бренд", "артикул", "партномер", "oem", "код", "модель", "размер",
             "цвет", "объем", "обьем", "вес", "тип", "материал", "количество"]

def build_text(name, attributes, max_attr_chars=260):
    parts = [str(name) if name is not None else ""]
    try:
        attrs = json.loads(attributes) if isinstance(attributes, str) else {}
    except Exception:
        attrs = {}
    if isinstance(attrs, dict) and attrs:
        low = {str(k).lower(): str(v) for k, v in attrs.items() if v}
        picked, used = [], set()
        for want in KEY_ORDER:
            for k, v in low.items():
                if want in k and k not in used:
                    picked.append(f"{k}:{v}"); used.add(k)
        rest = [f"{k}:{v}" for k, v in low.items() if k not in used]
        s = " ; ".join(picked + rest)[:max_attr_chars]
        parts.append(s)
    return " | ".join(parts)

t0 = time.time()
item_text, item_cat = {}, {}
f = pq.ParquetFile(ITEMS_PATH)
for b in f.iter_batches(columns=["id", "name", "attributes", "category"], batch_size=500_000):
    for i, n, a, c in b.to_pandas().itertuples(index=False, name=None):
        item_text[i] = build_text(n, a)
        item_cat[i] = c
print(f"items: {len(item_text):,} за {time.time()-t0:.0f}s")

## 2. Сплиты — ИДЕНТИЧНЫ нашим GBM-экспериментам
Групповые (union-find по товарам): manual seed 42 / 20%, LLM seed 13 / 3% компонент.

In [ ]:
def group_val_mask(df, val_frac, seed):
    parent = {}
    def find(x):
        p = parent.setdefault(x, x)
        while p != parent[p]:
            parent[p] = parent[parent[p]]; p = parent[p]
        parent[x] = p; return p
    for a, b in zip(df.id1.values, df.id2.values):
        ra, rb = find(a), find(b)
        if ra != rb: parent[rb] = ra
    comp = np.fromiter((find(i) for i in df.id1.values), dtype=np.int64, count=len(df))
    rng = np.random.RandomState(seed)
    uniq = np.unique(comp)
    val_set = set(uniq[rng.rand(len(uniq)) < val_frac].tolist())
    return np.fromiter((c in val_set for c in comp), dtype=bool, count=len(df))

m = pd.read_parquet(MATCHES_PATH)
m_val_mask = group_val_mask(m, 0.20, 42)
m_train, m_val = m[~m_val_mask].copy(), m[m_val_mask].copy()

ml = pd.read_parquet(MATCHES_LLM_PATH)
l_val_mask = group_val_mask(ml, 0.03, 13)
ml_train, ml_val = ml[~l_val_mask].copy(), ml[l_val_mask].copy()
ml_val = ml_val[(ml_val.target <= 0.2) | (ml_val.target >= 0.8)].copy()
ml_val["target"] = (ml_val.target >= 0.5).astype(int)

for df in (m_train, m_val, ml_train, ml_val):
    df["category"] = [item_cat[i] for i in df.id1]
del m, ml, m_val_mask, l_val_mask
import gc; gc.collect()
print(f"manual: {len(m_train):,}/{len(m_val):,}  llm: {len(ml_train):,}/{len(ml_val):,}")

## 3. Dataset / метрика

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PairDS(Dataset):
    def __init__(self, pairs_df):
        self.id1 = pairs_df.id1.values
        self.id2 = pairs_df.id2.values
        self.y = pairs_df.target.values.astype(np.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return item_text[self.id1[i]], item_text[self.id2[i]], self.y[i]

def collate(batch):
    t1, t2, y = zip(*batch)
    enc = tokenizer(list(t1), list(t2), padding=True, truncation=True,
                    max_length=MAX_LEN, return_tensors="pt")
    return enc, torch.tensor(y)

@torch.no_grad()
def predict(model, pairs_df, bs=512):
    model.eval()
    dl = DataLoader(PairDS(pairs_df), batch_size=bs, collate_fn=collate,
                    num_workers=0, shuffle=False)
    out = []
    for enc, _ in dl:
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(**enc).logits.squeeze(-1)
        out.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(out)

def macro_pr_auc(pairs_df, preds):
    z = pairs_df[["category", "target"]].copy(); z["pred"] = preds
    aps = z.groupby("category").apply(lambda g: average_precision_score(g.target, g.pred))
    return float(aps.mean()), aps

# быстрый eval-сэмпл, чтобы не гонять весь val каждые 20k шагов
ml_val_fast = ml_val.sample(min(60_000, len(ml_val)), random_state=0)

## 4. Стадия A — обучение на LLM-парах (soft labels)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1).to(device)

def train_stage(pairs_df, epochs, lr, tag):
    dl = DataLoader(PairDS(pairs_df), batch_size=BATCH, collate_fn=collate,
                    num_workers=0, shuffle=True, drop_last=True)
    steps_total = len(dl) * epochs
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, min(WARMUP, steps_total // 10), steps_total)
    scaler = torch.amp.GradScaler(enabled=False)  # bf16 не требует скейлинга
    lossf = nn.BCEWithLogitsLoss()
    step, t0, run_loss = 0, time.time(), 0.0
    best = 0.0
    for ep in range(epochs):
        for enc, y in dl:
            model.train()
            enc = {k: v.to(device, non_blocking=True) for k, v in enc.items()}
            y = y.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                logits = model(**enc).logits.squeeze(-1)
                loss = lossf(logits, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            run_loss += loss.item(); step += 1
            if step % 2000 == 0:
                sps = step * BATCH / (time.time() - t0)
                print(f"[{tag}] ep{ep} step {step}/{steps_total} loss={run_loss/2000:.4f} {sps:.0f} pair/s", flush=True)
                run_loss = 0.0
            if step % EVAL_EVERY == 0:
                mac, _ = macro_pr_auc(ml_val_fast, predict(model, ml_val_fast))
                print(f"[{tag}] step {step}: llm-val-fast Macro PR-AUC = {mac:.4f}", flush=True)
                torch.save({"model": model.state_dict(), "step": step, "metric": mac}, CKPT + ".last")
                if mac > best:
                    best = mac
                    torch.save({"model": model.state_dict(), "step": step, "metric": mac}, CKPT)
    return best

train_stage(ml_train, EPOCHS_A, LR_A, "A/llm")

## 5. Стадия B — дообучение на ручных парах

In [ ]:
print("stage B отключена (A/B: stage-A only лучше на LB)")

## 6. Финальные метрики — сравнивать с GBM: llm 0.608 / manual 0.625

In [ ]:
mac_l, aps_l = macro_pr_auc(ml_val, predict(model, ml_val))
print(f"[llm holdout, уверенные] Macro PR-AUC = {mac_l:.4f}")
print(aps_l.round(3).to_string())

mac_m, aps_m = macro_pr_auc(m_val, predict(model, m_val))
print(f"\n[manual holdout] Macro PR-AUC = {mac_m:.4f}")
print(aps_m.round(3).to_string())

## 7. Сохранение — скачать папку целиком и прислать для сборки сабмита

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
with open(f"{OUT_DIR}/metrics.json", "w") as fp:
    json.dump({"llm_holdout_macro_prauc": mac_l, "manual_holdout_macro_prauc": mac_m,
               "model": MODEL_NAME, "max_len": MAX_LEN,
               "llm_by_cat": aps_l.to_dict(), "manual_by_cat": aps_m.to_dict()}, fp,
              ensure_ascii=False, indent=1)
print("saved:", OUT_DIR)
print("модель сохранена на Google Drive:", OUT_DIR)